<span style="color:pink; font-size:15px;">File to run tests for the simulation code:</span>
<span style="color:yellow; font-size:15px;">Used to verify that the functions in `lin_alg.py` and `unitary.py` are working correctly, and test the overall simulation workflow.</span>

In [ ]:
import numpy as np
from state_gen import generate_state
from energy import energy, free_energy, passive_energy_g, passive_energy_l, ergotropy_gap, global_ergo, local_ergo
from local_to import qubit_ham, sho_ham, bath_gibbs_states, LTO_step, sys_gibbs_states, sanity_check_gibbs
from unitary import deg_unitary
from lin_alg import negativity, partial_trace, vn_entropy, rel_entropy, passive_state, partial_transpose
np.set_printoptions(linewidth=np.inf, precision=4, suppress=True)

def fmt(M): return np.array2string(M, precision=4, suppress_small=True, floatmode='fixed', max_line_width=80)

In [68]:
beta_a=1.0
beta_b=0.9
bath_dim = 20
w1 = w2 = 0.0001
omega_a = omega_b = 0.0001
a = 0.8
p = 0.5
n_terms = 4
N = np.sqrt(a * (1 - a))

kind_dict = {
    1: "product",
    2: "separable",
    3: "schmidt_ent",
    4: "pure_ent",
    5: "werner",
    6: "mixed_ent",
    7: "random"
}

In [69]:
rho12 = generate_state(kind=kind_dict[6], a=a, n_terms=n_terms, p=p)

neg_rho12 = negativity(rho12)
energy_rho12 = energy(rho12, w1, w2)
global_pass_energy_rho12 = passive_energy_g(rho12, w1, w2)
local_pass_energy_rho12 = passive_energy_l(rho12, w1, w2) 

print(f"rho12: \n{rho12}")
print(f"Negativity: {neg_rho12}")
print(f"Energy: {energy_rho12}")
print(f"Global Passive Energy: {global_pass_energy_rho12}")
print(f"Local Passive Energy: {local_pass_energy_rho12}")
print(f"Ergotropy Gap: {ergotropy_gap(rho12, w1, w2)}")

rho12: 
[[ 0.2801+0.j      0.013 +0.0278j  0.0016-0.1399j -0.0085+0.0716j]
 [ 0.013 -0.0278j  0.248 +0.j     -0.0941+0.0412j  0.0572+0.0907j]
 [ 0.0016+0.1399j -0.0941-0.0412j  0.3107+0.j      0.0163-0.0773j]
 [-0.0085-0.0716j  0.0572-0.0907j  0.0163+0.0773j  0.1612-0.j    ]]
Negativity: 0.018815546195826306
Energy: -2.3785353e-05
Global Passive Energy: -9.1746002e-05
Local Passive Energy: -3.7835122e-05
Ergotropy Gap: 5.391088e-05


In [70]:
gamma_a, gamma_b = bath_gibbs_states(beta_a, beta_b, bath_dim, omega_a=omega_a, omega_b=omega_b)
print(f"gamma_a: \n{gamma_a}")
print(f"gamma_b: \n{gamma_b}")

gamma_a: 
[[0.05+0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.05+0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.05+0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.  +0.j 0.05+0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.05+0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.  +0.j 0.05+0.j 0.  +0.j 0.  +0.j 0

In [71]:
sz = np.array([[-1, 0], [0, 1]], dtype=complex)
I = np.eye(2, dtype=complex)

Hs1 = qubit_ham(w1)
Hs2 = qubit_ham(w2)
Hb1 = sho_ham(dim=bath_dim, omega=omega_a)
Hb2 = sho_ham(dim=bath_dim, omega=omega_b)
Htot1 = np.kron(Hs1,np.eye(bath_dim)) + np.kron(np.eye(2),Hb1)
Htot2 = np.kron(Hs2,np.eye(bath_dim)) + np.kron(np.eye(2),Hb2)

print(f"Htot1: \n{np.diag(Htot1)}")
print(f"Htot2: \n{np.diag(Htot2)}")

Htot1: 
[-0.0001+0.j  0.    +0.j  0.0001+0.j  0.0002+0.j  0.0003+0.j  0.0004+0.j  0.0005+0.j  0.0006+0.j  0.0007+0.j  0.0008+0.j  0.0009+0.j  0.001 +0.j  0.0011+0.j  0.0012+0.j  0.0013+0.j  0.0014+0.j  0.0015+0.j  0.0016+0.j  0.0017+0.j  0.0018+0.j  0.0001+0.j  0.0002+0.j  0.0003+0.j  0.0004+0.j  0.0005+0.j  0.0006+0.j  0.0007+0.j  0.0008+0.j  0.0009+0.j  0.001 +0.j  0.0011+0.j  0.0012+0.j  0.0013+0.j  0.0014+0.j  0.0015+0.j  0.0016+0.j  0.0017+0.j  0.0018+0.j  0.0019+0.j  0.002 +0.j]
Htot2: 
[-0.0001+0.j  0.    +0.j  0.0001+0.j  0.0002+0.j  0.0003+0.j  0.0004+0.j  0.0005+0.j  0.0006+0.j  0.0007+0.j  0.0008+0.j  0.0009+0.j  0.001 +0.j  0.0011+0.j  0.0012+0.j  0.0013+0.j  0.0014+0.j  0.0015+0.j  0.0016+0.j  0.0017+0.j  0.0018+0.j  0.0001+0.j  0.0002+0.j  0.0003+0.j  0.0004+0.j  0.0005+0.j  0.0006+0.j  0.0007+0.j  0.0008+0.j  0.0009+0.j  0.001 +0.j  0.0011+0.j  0.0012+0.j  0.0013+0.j  0.0014+0.j  0.0015+0.j  0.0016+0.j  0.0017+0.j  0.0018+0.j  0.0019+0.j  0.002 +0.j]


In [72]:
Ua = deg_unitary(Hs1, Hb1, tol=1e-10)
Ub = deg_unitary(Hs2, Hb2, tol=1e-10)
# Ua = np.eye(bath_dim*2, dtype=complex)
# Ub = np.eye(bath_dim*2, dtype=complex)
print(f"Ua: \n{Ua}")
print(f"Ub: \n{Ub}")

Degenerate blocks: E=0.0001: [2, 20], E=0.0002: [3, 21], E=0.0003: [4, 22], E=0.0004: [5, 23], E=0.0005: [6, 24], E=0.0006: [7, 25], E=0.0007: [8, 26], E=0.0008: [9, 27], E=0.0009: [10, 28], E=0.0010: [11, 29], E=0.0011: [12, 30], E=0.0012: [13, 31], E=0.0013: [14, 32], E=0.0014: [15, 33], E=0.0015: [16, 34], E=0.0016: [17, 35], E=0.0017: [18, 36], E=0.0018: [19, 37]
Degenerate blocks: E=0.0001: [2, 20], E=0.0002: [3, 21], E=0.0003: [4, 22], E=0.0004: [5, 23], E=0.0005: [6, 24], E=0.0006: [7, 25], E=0.0007: [8, 26], E=0.0008: [9, 27], E=0.0009: [10, 28], E=0.0010: [11, 29], E=0.0011: [12, 30], E=0.0012: [13, 31], E=0.0013: [14, 32], E=0.0014: [15, 33], E=0.0015: [16, 34], E=0.0016: [17, 35], E=0.0017: [18, 36], E=0.0018: [19, 37]
Ua: 
[[-0.7745-0.6326j  0.    +0.j      0.    +0.j     ...  0.    +0.j      0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j     -0.5993+0.8005j  0.    +0.j     ...  0.    +0.j      0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j     -0.4701+0.1221j

In [73]:
# Manual check of degeneracy and unitarity
print(np.max(np.abs(Ua @ Htot1 - Htot1 @ Ua) < 1e-10))
print(np.max(np.abs(Ub @ Htot2 - Htot2 @ Ub) < 1e-10))
print(np.max(np.abs(Ub @ Htot1 - Htot1 @ Ub) < 1e-10))
print(np.allclose(Ua @ Ua.conj().T,np.eye(Ua.shape[0]),atol=1e-10))
print(np.allclose(Ub @ Ub.conj().T,np.eye(Ub.shape[0]),atol=1e-10))

True
True
True
True
True


In [74]:
sigma, gamma_a1, gamma_b1 = LTO_step(rho12, gamma_a, gamma_b, Ua, Ub)

print(f"rho12: \n{rho12}")
print(f"sigma: \n{sigma}\n")

rho12: 
[[ 0.2801+0.j      0.013 +0.0278j  0.0016-0.1399j -0.0085+0.0716j]
 [ 0.013 -0.0278j  0.248 +0.j     -0.0941+0.0412j  0.0572+0.0907j]
 [ 0.0016+0.1399j -0.0941-0.0412j  0.3107+0.j      0.0163-0.0773j]
 [-0.0085-0.0716j  0.0572-0.0907j  0.0163+0.0773j  0.1612-0.j    ]]
sigma: 
[[ 0.2578+0.j     -0.002 +0.0013j -0.0015-0.006j  -0.0003-0.0008j]
 [-0.002 -0.0013j  0.2445+0.j      0.    -0.0012j  0.0024-0.0036j]
 [-0.0015+0.006j   0.    +0.0012j  0.2562+0.j     -0.0023+0.002j ]
 [-0.0003+0.0008j  0.0024+0.0036j -0.0023-0.002j   0.2415+0.j    ]]



In [75]:
rho1 = partial_trace(rho12, sys=1)
rho2 = partial_trace(rho12, sys=0)
pass_rho_1 = passive_state(rho1)
pass_rho_2 = passive_state(rho2)

print(f"Partial trace of rho12 (sys=1): \n{rho1}")
print(f"Partial trace of rho12 (sys=0): \n{rho2}\n")
# print(f"Passive state of rho1: \n{pass_rho_1}")
# print(f"Passive state of rho2: \n{pass_rho_2}")

sigma1 = partial_trace(sigma, sys=1)
sigma2 = partial_trace(sigma, sys=0)
pass_sigma_1 = passive_state(sigma1)
pass_sigma_2 = passive_state(sigma2)

print(f"Partial trace of sigma (sys=1): \n{sigma1}")
print(f"Partial trace of sigma (sys=0): \n{sigma2}\n")
# print(f"Passive state of sigma1: \n{pass_sigma_1}")
# print(f"Passive state of sigma2: \n{pass_sigma_2}")

gibbs_a, gibbs_b = sys_gibbs_states(beta_a, beta_b, w1, w2)
print(f"Gibbs state of system A: \n{gibbs_a}")
print(f"Gibbs state of system B: \n{gibbs_b}")

print(np.allclose(sigma, np.kron(sigma1, sigma2), atol=1e-10))
print(np.allclose(rho12, np.kron(rho1, rho2)))

Partial trace of rho12 (sys=1): 
[[0.5281+0.j     0.0588-0.0492j]
 [0.0588+0.0492j 0.4719+0.j    ]]
Partial trace of rho12 (sys=0): 
[[0.5908+0.j     0.0294-0.0495j]
 [0.0294+0.0495j 0.4092+0.j    ]]

Partial trace of sigma (sys=1): 
[[0.5023+0.j     0.0008-0.0096j]
 [0.0008+0.0096j 0.4977+0.j    ]]
Partial trace of sigma (sys=0): 
[[ 0.514 +0.j     -0.0043+0.0034j]
 [-0.0043-0.0034j  0.486 +0.j    ]]

Gibbs state of system A: 
[[0.5-0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Gibbs state of system B: 
[[0.5-0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
False
False


In [76]:
sanity_check_gibbs(beta_a=beta_a, beta_b=beta_b, w1=w1, w2=w2, omega=omega_a, bath_dim=bath_dim, tol=1e-8)

SANITY CHECK: Gibbs fixed point
Degenerate blocks: E=0.0001: [2, 20], E=0.0002: [3, 21], E=0.0003: [4, 22], E=0.0004: [5, 23], E=0.0005: [6, 24], E=0.0006: [7, 25], E=0.0007: [8, 26], E=0.0008: [9, 27], E=0.0009: [10, 28], E=0.0010: [11, 29], E=0.0011: [12, 30], E=0.0012: [13, 31], E=0.0013: [14, 32], E=0.0014: [15, 33], E=0.0015: [16, 34], E=0.0016: [17, 35], E=0.0017: [18, 36], E=0.0018: [19, 37]
Degenerate blocks: E=0.0001: [2, 20], E=0.0002: [3, 21], E=0.0003: [4, 22], E=0.0004: [5, 23], E=0.0005: [6, 24], E=0.0006: [7, 25], E=0.0007: [8, 26], E=0.0008: [9, 27], E=0.0009: [10, 28], E=0.0010: [11, 29], E=0.0011: [12, 30], E=0.0012: [13, 31], E=0.0013: [14, 32], E=0.0014: [15, 33], E=0.0015: [16, 34], E=0.0016: [17, 35], E=0.0017: [18, 36], E=0.0018: [19, 37]
  rho12 preserved:   PASS  (max err = 2.78e-17)
  gamma_Ra preserved: PASS  (max err = 6.25e-17)
  gamma_Rb preserved: PASS  (max err = 3.47e-17)
  trace(sigma12)=1:  PASS  (err = 3.44e-20)
  ΔF ≤ 0:            PASS  (ΔF = 0.00e

True

In [77]:
# ── Precompute everything ────────────────────────────────────────
E_rho   = energy(rho12, w1, w2)
E_sigma = energy(sigma, w1, w2)

Eg_rho   = global_ergo(rho12, w1, w2)
Eg_sigma = global_ergo(sigma, w1, w2)
El_rho   = local_ergo(rho12, w1, w2)
El_sigma = local_ergo(sigma, w1, w2)

F1_before = free_energy(rho1,   Hs1, beta_a)
F1_after  = free_energy(sigma1, Hs1, beta_a)
F2_before = free_energy(rho2,   Hs2, beta_b)
F2_after  = free_energy(sigma2, Hs2, beta_b)

F1_gibbs  = free_energy(gibbs_a, Hs1, beta_a)
F2_gibbs  = free_energy(gibbs_b, Hs2, beta_b)

def E_sys(rho, w):   return -w*rho[0,0].real + w*rho[1,1].real
def E_bath(rho, om): return np.trace(sho_ham(dim=bath_dim, omega=om) @ rho).real

# ── Entanglement ─────────────────────────────────────────────────
print("─"*50)
print("  ENTANGLEMENT")
print(f"  Negativity  ρ → σ :  {negativity(rho12):.6f}  →  {negativity(sigma):.6f}")

# ── Energy ───────────────────────────────────────────────────────
print("─"*50)
print("  ENERGY")
print(f"  E(ρ) → E(σ) :  {E_rho:.6f}  →  {E_sigma:.6f}   ΔE = {E_sigma - E_rho:.2e}")

# ── Ergotropy ────────────────────────────────────────────────────
print("─"*50)
print("  ERGOTROPY")
print(f"  Global  ρ → σ :  {Eg_rho:.6f}  →  {Eg_sigma:.6f}   Δ = {Eg_sigma - Eg_rho:.2e}")
print(f"  Local   ρ → σ :  {El_rho:.6f}  →  {El_sigma:.6f}   Δ = {El_sigma - El_rho:.2e}")
print(f"  Gap     ρ → σ :  {ergotropy_gap(rho12,w1,w2):.6f}  →  {ergotropy_gap(sigma,w1,w2):.6f}")

# ── Subsystem ergotropy ──────────────────────────────────────────
print("─"*50)
print("  SUBSYSTEM ERGOTROPY (Local)")
R1_before = E_sys(rho1,   w1) - E_sys(pass_rho_1,  w1)
R1_after  = E_sys(sigma1, w1) - E_sys(pass_sigma_1, w1)
R2_before = E_sys(rho2,   w2) - E_sys(pass_rho_2,   w2)
R2_after  = E_sys(sigma2, w2) - E_sys(pass_sigma_2, w2)
print(f"  R₁  ρ₁ → σ₁ :  {R1_before:.6f}  →  {R1_after:.6f}   Δ = {R1_after - R1_before:.2e}")
print(f"  R₂  ρ₂ → σ₂ :  {R2_before:.6f}  →  {R2_after:.6f}   Δ = {R2_after - R2_before:.2e}")

# ── Analytical bounds ────────────────────────────────────────────
print("─"*50)
print("  ANALYTICAL BOUNDS  vs  ACTUAL GAPS (Local)")
bound1  = rel_entropy(pass_rho_1, gibbs_a) / beta_a
bound2  = rel_entropy(pass_rho_2, gibbs_b) / beta_b
print(f"  Subsystem 1:  bound = {bound1:.6f}   actual ΔR₁ = {R1_after - R1_before:.6f}")
print(f"  Subsystem 2:  bound = {bound2:.6f}   actual ΔR₂ = {R2_after - R2_before:.6f}")
print(f"  Total Local bound  = {bound1 + bound2:.6f}   actual ΔR_local = {El_sigma - El_rho:.6f}")

print("─"*50)
print("  ANALYTICAL BOUNDS  vs  ACTUAL GAPS (Global)")
local_term = (1/beta_a) * (vn_entropy(sigma1) - vn_entropy(rho1)) + (1/beta_b) * (vn_entropy(sigma2) - vn_entropy(rho2))
energy_term = energy(passive_state(rho12), w1, w2) - energy(passive_state(sigma), w1, w2)
print(f"  Total Global bound  = {local_term + energy_term:.6f}   actual ΔR_global = {Eg_sigma - Eg_rho:.6f}")

# ── Free energy ──────────────────────────────────────────────────
print("─"*50)
print("  FREE ENERGY")
print(f"  F₁  ρ₁ → σ₁ :  {F1_before:.6f}  →  {F1_after:.6f}   ΔF₁ = {F1_after - F1_before:.2e}")
print(f"  F₂  ρ₂ → σ₂ :  {F2_before:.6f}  →  {F2_after:.6f}   ΔF₂ = {F2_after - F2_before:.2e}")
print(f"  F₁ - F₁_gibbs :  {F1_before - F1_gibbs:.2e}  →  {F1_after - F1_gibbs:.2e}")
print(f"  F₂ - F₂_gibbs :  {F2_before - F2_gibbs:.2e}  →  {F2_after - F2_gibbs:.2e}")
print(f"  Total F - F_gibbs :  {(F1_before - F1_gibbs) + (F2_before - F2_gibbs):.2e}  →  {(F1_after - F1_gibbs) + (F2_after - F2_gibbs):.2e}")

# ── Relative entropy ─────────────────────────────────────────────
print("─"*50)
print("  RELATIVE ENTROPY  S(ρ'||γ')/β'")
print(f"  S(ρ₁||γ_a)/β_a :  {rel_entropy(rho1,   gibbs_a)/beta_a:.6f}  →  {rel_entropy(sigma1, gibbs_a)/beta_a:.6f}")
print(f"  S(ρ₂||γ_b)/β_b :  {rel_entropy(rho2,   gibbs_b)/beta_b:.6f}  →  {rel_entropy(sigma2, gibbs_b)/beta_b:.6f}")

# ── Total energy conservation ────────────────────────────────────
print("─"*50)
print("  TOTAL ENERGY CONSERVATION  (sys + bath)")
E_tot1_before = E_sys(rho1,   w1) + E_bath(gamma_a,  omega_a)
E_tot1_after  = E_sys(sigma1, w1) + E_bath(gamma_a1, omega_a)
E_tot2_before = E_sys(rho2,   w2) + E_bath(gamma_b,  omega_b)
E_tot2_after  = E_sys(sigma2, w2) + E_bath(gamma_b1, omega_b)
print(f"  E₁_tot  before → after :  {E_tot1_before:.6f}  →  {E_tot1_after:.6f}   Δ = {E_tot1_after - E_tot1_before:.2e}")
print(f"  E₂_tot  before → after :  {E_tot2_before:.6f}  →  {E_tot2_after:.6f}   Δ = {E_tot2_after - E_tot2_before:.2e}")
print("─"*50)

──────────────────────────────────────────────────
  ENTANGLEMENT
  Negativity  ρ → σ :  0.018816  →  0.000000
──────────────────────────────────────────────────
  ENERGY
  E(ρ) → E(σ) :  -0.000024  →  -0.000003   ΔE = 2.05e-05
──────────────────────────────────────────────────
  ERGOTROPY
  Global  ρ → σ :  0.000068  →  0.000002   Δ = -6.60e-05
  Local   ρ → σ :  0.000014  →  0.000002   Δ = -1.23e-05
  Gap     ρ → σ :  0.000054  →  0.000000
──────────────────────────────────────────────────
  SUBSYSTEM ERGOTROPY (Local)
  R₁  ρ₁ → σ₁ :  0.000011  →  0.000002   Δ = -9.18e-06
  R₂  ρ₂ → σ₂ :  0.000003  →  0.000000   Δ = -3.14e-06
──────────────────────────────────────────────────
  ANALYTICAL BOUNDS  vs  ACTUAL GAPS (Local)
  Subsystem 1:  bound = 0.013372   actual ΔR₁ = -0.000009
  Subsystem 2:  bound = 0.025880   actual ΔR₂ = -0.000003
  Total Local bound  = 0.039252   actual ΔR_local = -0.000012
──────────────────────────────────────────────────
  ANALYTICAL BOUNDS  vs  ACTUAL GAPS (